# Bitget `market` -- gap check (UTA)

Same rationale as `market/classic.ipynb`: production has no `Market`/`PerpMarket`
implementation, but `uta.market`/`uta.trade`/`uta.position` are real, confirmed trading
surfaces. Split by account mode for the same reason as `earn`/`wallet`/`reporting`:
UTA's `place`/`cancel`/`unfilled`/`fills` all key off one `category` (`SPOT`, `MARGIN`,
`USDT-FUTURES`, `COIN-FUTURES`, `USDC-FUTURES`) against ONE unified collateral pool
(`account.assets()`), a structurally different trading/collateral model than Classic's
per-product-line balances -- not just a credential swap.

In [1]:
import asyncio
import os
from decimal import Decimal
from datetime import datetime, timedelta, timezone

from typing_extensions import cast

from typed_bitget import Bitget
from typed_bitget.core import timestamp_millis
from typed_bitget.uta.trade.order.place import LimitOrderRequest, MarketOrderRequest
from dotenv import load_dotenv

from tribulnation.sdk.market import (
  Book,
  Collateral,
  FundingPayment,
  FundingRate,
  NextFunding,
  Order,
  OrderResponse,
  OrderState,
  PerpCollateral,
  PerpPosition,
  Position,
  Rules,
  Settings,
  Trade,
)

load_dotenv()

client = await Bitget.new(
  access_key=os.environ['BITGET_UTA_ACCESS_KEY'],
  secret_key=os.environ['BITGET_UTA_SECRET_KEY'],
  passphrase=os.environ['BITGET_UTA_PASSPHRASE'],
).__aenter__()

SPOT_MARKETS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
PERP_MARKETS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
PERP_CATEGORY = 'USDT-FUTURES'

## `Market` (UTA, category `SPOT`)

In [2]:
async def depth(symbol: str, *, levels: int | None = None) -> Book:
  raw = await client.uta.market.orderbook(category='SPOT', symbol=symbol, limit=levels)
  return Book(
    bids=[Book.Entry(Decimal(str(p)), Decimal(str(q))) for p, q in raw['b']],
    asks=[Book.Entry(Decimal(str(p)), Decimal(str(q))) for p, q in raw['a']],
  )

{symbol: await depth(symbol, levels=5) for symbol in SPOT_MARKETS}

{'BTCUSDT': Book(bids=[Book.Entry(price=Decimal('80924.99'), qty=Decimal('1.728128')), Book.Entry(price=Decimal('80924.91'), qty=Decimal('0.000018')), Book.Entry(price=Decimal('80924.85'), qty=Decimal('0.054291')), Book.Entry(price=Decimal('80924.84'), qty=Decimal('0.037067')), Book.Entry(price=Decimal('80924.6'), qty=Decimal('0.001235'))], asks=[Book.Entry(price=Decimal('80925.0'), qty=Decimal('0.019585')), Book.Entry(price=Decimal('80925.16'), qty=Decimal('0.003556')), Book.Entry(price=Decimal('80927.65'), qty=Decimal('0.004077')), Book.Entry(price=Decimal('80928.24'), qty=Decimal('0.001')), Book.Entry(price=Decimal('80929.07'), qty=Decimal('0.011542'))]),
 'ETHUSDT': Book(bids=[Book.Entry(price=Decimal('2520.33'), qty=Decimal('13.723')), Book.Entry(price=Decimal('2520.3'), qty=Decimal('0.9748')), Book.Entry(price=Decimal('2520.28'), qty=Decimal('0.3968')), Book.Entry(price=Decimal('2520.19'), qty=Decimal('2.0747')), Book.Entry(price=Decimal('2520.18'), qty=Decimal('0.7935'))], asks=

`rules()` below leaves `maker_fee`/`taker_fee` at `0`: unlike Classic's `SpotSymbol`,
`uta.market.instruments`'s spot rows carry no fee-rate fields at all. Both are declared
`NotRequired` and documented "futures only", and the live `SPOT` rows for all three
symbols below confirm it -- they carry `pricePrecision`/`quantityPrecision`/
`minOrderQty`/`minOrderAmount`/`status` and no `makerFeeRate`/`takerFeeRate`, while the
same call for `USDT-FUTURES` returns both. No alternative UTA spot-fee-rate endpoint was
found in scope.

In [3]:
async def rules(symbol: str, *, refetch: bool = False) -> Rules:
  raw = await client.uta.market.instruments(category='SPOT', symbol=symbol)
  s = raw[0]
  return Rules(
    base=s['baseCoin'],
    quote=s['quoteCoin'],
    fee_asset=s['quoteCoin'],
    tick_size=Decimal(10) ** -int(s['pricePrecision']),
    step_size=Decimal(10) ** -int(s['quantityPrecision']),
    fixed_min_qty=s['minOrderQty'],
    min_value=s['minOrderAmount'],
    max_qty=s['maxOrderQty'],
    maker_fee=s.get('makerFeeRate') or Decimal(0),
    taker_fee=s.get('takerFeeRate') or Decimal(0),
    api=s['status'] in ('listed', 'online'),
    details=s,
  )

{symbol: await rules(symbol) for symbol in SPOT_MARKETS}

{'BTCUSDT': Rules(base='BTC', quote='USDT', fee_asset='USDT', tick_size=Decimal('0.01'), step_size=Decimal('0.000001'), fixed_min_qty=Decimal('0.000001'), min_value=Decimal('1'), max_qty=Decimal('0'), fixed_min_price=None, rel_min_price=None, rel_max_price=None, fixed_max_price=None, maker_fee=Decimal('0'), taker_fee=Decimal('0'), api=True, details={'symbol': 'BTCUSDT', 'category': 'SPOT', 'baseCoin': 'BTC', 'quoteCoin': 'USDT', 'buyLimitPriceRatio': Decimal('0.02'), 'sellLimitPriceRatio': Decimal('0.02'), 'minOrderQty': Decimal('0.000001'), 'maxOrderQty': Decimal('0'), 'pricePrecision': 2, 'quantityPrecision': 6, 'quotePrecision': '8', 'minOrderAmount': Decimal('1'), 'maxSymbolOrderNum': '', 'maxProductOrderNum': 400, 'status': 'online', 'maintainTime': '', 'isReality': 'no', 'symbolType': 'crypto', 'maxPositionNum': 200, 'launchTime': datetime.datetime(2018, 7, 24, 17, 46, tzinfo=datetime.timezone.utc)}),
 'ETHUSDT': Rules(base='ETH', quote='USDT', fee_asset='USDT', tick_size=Decimal

In [4]:
async def open_orders(symbol: str) -> list[OrderState]:
  raw = await client.uta.trade.order.unfilled(category='SPOT', symbol=symbol)
  out: list[OrderState] = []
  for o in raw['list'] or []:
    qty = Decimal(o['qty'])
    filled = Decimal(o['cumExecQty'])
    sign = 1 if o['side'] == 'buy' else -1
    out.append(OrderState(
      id=o['orderId'],
      price=Decimal(o['price']),
      qty=sign * qty,
      filled_qty=sign * filled,
      active=True,  # this endpoint only lists currently-unfilled orders
      details=o,
    ))
  return out

{symbol: await open_orders(symbol) for symbol in SPOT_MARKETS}

{'BTCUSDT': [], 'ETHUSDT': [], 'SOLUSDT': []}

In [5]:
async def trades_history(symbol: str, start: datetime, end: datetime) -> list[Trade]:
  raw = await client.uta.trade.order.fills(
    category='SPOT', start_time=start, end_time=end,
  )
  out: list[Trade] = []
  for f in raw['list'] or []:
    if f['symbol'] != symbol:
      continue
    qty = Decimal(f['execQty'])
    fee = None
    if f['feeDetail']:
      fee_amount = sum(abs(Decimal(d['fee'])) for d in f['feeDetail'])
      if fee_amount:
        fee = Trade.Fee(amount=fee_amount, asset=f['feeDetail'][0]['feeCoin'])
    out.append(Trade(
      id=f['execId'],
      price=Decimal(f['execPrice']),
      qty=qty if f['side'] == 'buy' else -qty,
      time=f['createdTime'],
      maker=f['tradeScope'].lower() == 'maker',
      fee=fee,
      details=f,
    ))
  return out

end = datetime.now(timezone.utc)
start = end - timedelta(hours=24)
{symbol: await trades_history(symbol, start, end) for symbol in SPOT_MARKETS}

{'BTCUSDT': [], 'ETHUSDT': [], 'SOLUSDT': []}

`position()`/`available_notional()` both read the *same* `account.assets()` call UTA
uses for its overall collateral (see `reporting/uta.ipynb`'s `Snapshots` section) --
there's no separate "spot balance" endpoint the way Classic has `spot.account.assets`,
because UTA doesn't partition spot balances from the rest of the unified account.

In [6]:
async def coin_balance(coin: str) -> Decimal:
  raw = await client.uta.account.assets()
  row = next((a for a in raw['assets'] if a['coin'] == coin), None)
  return row['balance'] if row else Decimal(0)


async def position(symbol: str) -> Position:
  base = symbol.removesuffix('USDT')  # SPOT_MARKETS are all *USDT pairs here
  return Position(size=await coin_balance(base))

{symbol: await position(symbol) for symbol in SPOT_MARKETS}

{'BTCUSDT': Position(size=Decimal('0')),
 'ETHUSDT': Position(size=Decimal('0')),
 'SOLUSDT': Position(size=Decimal('0'))}

In [7]:
async def available_notional(symbol: str) -> Decimal:
  return await coin_balance('USDT')

{symbol: await available_notional(symbol) for symbol in SPOT_MARKETS}

{'BTCUSDT': Decimal('11.07345081'),
 'ETHUSDT': Decimal('11.07345081'),
 'SOLUSDT': Decimal('11.07345081')}

In [ ]:
async def place_order(symbol: str, order: Order, *, settings: Settings = {}) -> OrderResponse:
  qty_signed = Decimal(str(order['qty']))
  side = 'buy' if qty_signed > 0 else 'sell'
  qty = abs(qty_signed)
  price = Decimal(str(order['price']))
  body: LimitOrderRequest | MarketOrderRequest
  if order['type'] == 'MARKET':
    body = {'category': 'SPOT', 'symbol': symbol, 'side': side, 'qty': qty, 'orderType': 'market'}
  else:
    time_in_force = 'post_only' if order['type'] == 'POST_ONLY' else 'gtc'
    body = {
      'category': 'SPOT', 'symbol': symbol, 'side': side, 'qty': qty,
      'orderType': 'limit', 'price': price, 'timeInForce': time_in_force,
    }
  raw = await client.uta.trade.order.place(body)
  return OrderResponse(id=raw['orderId'], details=raw)

# Not executed here -- would place a real order on the account.
await place_order('BTCUSDT', {'qty': Decimal('0.0001'), 'price': Decimal('20000'), 'type': 'LIMIT'})

In [ ]:
async def cancel_order(symbol: str, id: str, *, settings: Settings = {}):
  return await client.uta.trade.order.cancel(order_id=id, category='SPOT')

# Not executed here -- would cancel a real order on the account.
await cancel_order('BTCUSDT', '123456')

### Coverage assessment: `Market` (UTA spot)

**Mostly supported.** `depth`, `open_orders`, `trades_history`, `position`,
`available_notional`, `place_order`, `cancel_order` all map cleanly (confirmed against
real live responses: the unified account holds USDT/USDC/XRP/BGB dust, has no unfilled
spot orders, and no spot fills in the last 24 hours). The one real gap:
`rules().maker_fee`/`taker_fee` have no source under `uta.market.instruments` for `SPOT`
rows (see the note above) -- left at `0` rather than guessed. `collateral()` is left
unimplemented, same choice as Classic's spot section.

## `PerpMarket` (UTA, category `USDT-FUTURES`)

In [8]:
async def perp_depth(symbol: str, *, levels: int | None = None) -> Book:
  raw = await client.uta.market.orderbook(category=PERP_CATEGORY, symbol=symbol, limit=levels)
  return Book(
    bids=[Book.Entry(Decimal(str(p)), Decimal(str(q))) for p, q in raw['b']],
    asks=[Book.Entry(Decimal(str(p)), Decimal(str(q))) for p, q in raw['a']],
  )

{symbol: await perp_depth(symbol, levels=5) for symbol in PERP_MARKETS}

{'BTCUSDT': Book(bids=[Book.Entry(price=Decimal('80893.9'), qty=Decimal('4.8769')), Book.Entry(price=Decimal('80893.2'), qty=Decimal('0.0001')), Book.Entry(price=Decimal('80893.1'), qty=Decimal('0.0002')), Book.Entry(price=Decimal('80893.0'), qty=Decimal('0.0249')), Book.Entry(price=Decimal('80892.8'), qty=Decimal('0.0091'))], asks=[Book.Entry(price=Decimal('80894.0'), qty=Decimal('0.0498')), Book.Entry(price=Decimal('80895.3'), qty=Decimal('0.0001')), Book.Entry(price=Decimal('80895.9'), qty=Decimal('0.0001')), Book.Entry(price=Decimal('80896.1'), qty=Decimal('0.0001')), Book.Entry(price=Decimal('80896.9'), qty=Decimal('0.0001'))]),
 'ETHUSDT': Book(bids=[Book.Entry(price=Decimal('2519.23'), qty=Decimal('60.68')), Book.Entry(price=Decimal('2519.2'), qty=Decimal('1.99')), Book.Entry(price=Decimal('2519.15'), qty=Decimal('1.49')), Book.Entry(price=Decimal('2519.13'), qty=Decimal('2.47')), Book.Entry(price=Decimal('2519.11'), qty=Decimal('0.01'))], asks=[Book.Entry(price=Decimal('2519.24

In [9]:
async def perp_rules(symbol: str, *, refetch: bool = False) -> Rules:
  raw = await client.uta.market.instruments(category=PERP_CATEGORY, symbol=symbol)
  s = raw[0]
  return Rules(
    base=s['baseCoin'],
    quote=s['quoteCoin'],
    fee_asset=s['quoteCoin'],
    tick_size=Decimal(10) ** -int(s['pricePrecision']),
    step_size=Decimal(10) ** -int(s['quantityPrecision']),
    fixed_min_qty=s['minOrderQty'],
    min_value=s['minOrderAmount'],
    max_qty=s['maxOrderQty'],
    maker_fee=s.get('makerFeeRate') or Decimal(0),
    taker_fee=s.get('takerFeeRate') or Decimal(0),
    api=s['status'] in ('listed', 'online'),
    details=s,
  )

{symbol: await perp_rules(symbol) for symbol in PERP_MARKETS}

{'BTCUSDT': Rules(base='BTC', quote='USDT', fee_asset='USDT', tick_size=Decimal('0.1'), step_size=Decimal('0.0001'), fixed_min_qty=Decimal('0.0001'), min_value=Decimal('5'), max_qty=Decimal('1200'), fixed_min_price=None, rel_min_price=None, rel_max_price=None, fixed_max_price=None, maker_fee=Decimal('0.0002'), taker_fee=Decimal('0.0006'), api=True, details={'symbol': 'BTCUSDT', 'category': 'USDT-FUTURES', 'baseCoin': 'BTC', 'quoteCoin': 'USDT', 'buyLimitPriceRatio': Decimal('0.05'), 'sellLimitPriceRatio': Decimal('0.05'), 'feeRateUpRatio': Decimal('0.005'), 'minOrderQty': Decimal('0.0001'), 'maxOrderQty': Decimal('1200'), 'maxMarketOrderQty': Decimal('220'), 'pricePrecision': 1, 'quantityPrecision': 4, 'quotePrecision': '', 'minOrderAmount': Decimal('5'), 'maxSymbolOrderNum': '', 'maxProductOrderNum': 400, 'status': 'online', 'offTime': datetime.datetime(1969, 12, 31, 23, 59, 59, 999000, tzinfo=datetime.timezone.utc), 'limitOpenTime': datetime.datetime(1969, 12, 31, 23, 59, 59, 999000,

In [10]:
async def perp_open_orders(symbol: str) -> list[OrderState]:
  raw = await client.uta.trade.order.unfilled(category=PERP_CATEGORY, symbol=symbol)
  out: list[OrderState] = []
  for o in raw['list'] or []:
    qty = Decimal(o['qty'])
    filled = Decimal(o['cumExecQty'])
    sign = 1 if o['side'] == 'buy' else -1
    out.append(OrderState(
      id=o['orderId'],
      price=Decimal(o['price']),
      qty=sign * qty,
      filled_qty=sign * filled,
      active=True,
      details=o,
    ))
  return out

{symbol: await perp_open_orders(symbol) for symbol in PERP_MARKETS}

{'BTCUSDT': [], 'ETHUSDT': [], 'SOLUSDT': []}

In [11]:
async def perp_trades_history(symbol: str, start: datetime, end: datetime) -> list[Trade]:
  raw = await client.uta.trade.order.fills(
    category=PERP_CATEGORY, start_time=start, end_time=end,
  )
  out: list[Trade] = []
  for f in raw['list'] or []:
    if f['symbol'] != symbol:
      continue
    qty = Decimal(f['execQty'])
    fee = None
    if f['feeDetail']:
      fee_amount = sum(abs(Decimal(d['fee'])) for d in f['feeDetail'])
      if fee_amount:
        fee = Trade.Fee(amount=fee_amount, asset=f['feeDetail'][0]['feeCoin'])
    out.append(Trade(
      id=f['execId'],
      price=Decimal(f['execPrice']),
      qty=qty if f['side'] == 'buy' else -qty,
      time=f['createdTime'],
      maker=f['tradeScope'].lower() == 'maker',
      fee=fee,
      details=f,
    ))
  return out

{symbol: await perp_trades_history(symbol, start, end) for symbol in PERP_MARKETS}

{'BTCUSDT': [], 'ETHUSDT': [], 'SOLUSDT': []}

In [12]:
async def index(symbol: str, *, settings: Settings = {}) -> Decimal:
  raw = await client.uta.market.tickers(category=PERP_CATEGORY, symbol=symbol)
  t = raw[0]
  return t.get('indexPrice') or t['lastPrice']

{symbol: await index(symbol) for symbol in PERP_MARKETS}

{'BTCUSDT': Decimal('80929.1345'),
 'ETHUSDT': Decimal('2520.411'),
 'SOLUSDT': Decimal('103.8355')}

In [2]:
async def next_funding(symbol: str) -> NextFunding:
  # Read unvalidated: `CurrentFundingRate.cashDividendNextUpdate` is typed
  # `NotRequired[TimestampMillis | None]`, but the live API sends the literal *string*
  # `'null'` -- re-confirmed today for all three symbols here, none of which is a
  # Reality-stock symbol (`isReality: 'no'`), contrary to the field's own docstring. No
  # epoch-millis parse accepts that, so validation rejects every row -- a real
  # typed_bitget bug. The fields this maps are converted by hand instead: unvalidated rows
  # carry the wire values, so `nextUpdate` is still the epoch-millis string `parse` wants,
  # not the `datetime` the validated type promises.
  raw = await client.uta.market.funding_rate.current(
    category=PERP_CATEGORY, symbol=symbol, validate=False,
  )
  r = raw[0]
  return NextFunding(
    rate=Decimal(r['fundingRate']),
    time=timestamp_millis.parse(cast(str, r['nextUpdate'])),
    interval=timedelta(hours=int(r['fundingRateInterval'])),
  )

{symbol: await next_funding(symbol) for symbol in PERP_MARKETS}

{'BTCUSDT': NextFunding(rate=Decimal('0.000092'), time=datetime.datetime(2026, 9, 4, 16, 0, tzinfo=datetime.timezone.utc), premium=None, interval=datetime.timedelta(seconds=28800)),
 'ETHUSDT': NextFunding(rate=Decimal('0.000075'), time=datetime.datetime(2026, 9, 4, 16, 0, tzinfo=datetime.timezone.utc), premium=None, interval=datetime.timedelta(seconds=28800)),
 'SOLUSDT': NextFunding(rate=Decimal('0.000038'), time=datetime.datetime(2026, 9, 4, 16, 0, tzinfo=datetime.timezone.utc), premium=None, interval=datetime.timedelta(seconds=28800))}

In [14]:
async def funding_rates(
  symbol: str, start: datetime | None = None, end: datetime | None = None,
) -> list[FundingRate]:
  # `funding_rate.history`'s `cursor` behaves as a page number, not a date range -- one
  # page is fetched and filtered client-side, same caveat as `market/classic.ipynb`.
  raw = await client.uta.market.funding_rate.history(category=PERP_CATEGORY, symbol=symbol, limit=50)
  out = [FundingRate(rate=r['fundingRate'], time=r['fundingRateTimestamp']) for r in raw['resultList']]
  if start is not None:
    out = [r for r in out if r.time >= start]
  if end is not None:
    out = [r for r in out if r.time <= end]
  return out

funding_end = datetime.now(timezone.utc)
funding_start = funding_end - timedelta(days=7)
{symbol: await funding_rates(symbol, funding_start, funding_end) for symbol in PERP_MARKETS}

{'BTCUSDT': [FundingRate(rate=Decimal('0.000079'), time=datetime.datetime(2026, 9, 4, 8, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000008'), time=datetime.datetime(2026, 9, 4, 0, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000043'), time=datetime.datetime(2026, 9, 3, 16, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000094'), time=datetime.datetime(2026, 9, 3, 8, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.00004'), time=datetime.datetime(2026, 9, 3, 0, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000069'), time=datetime.datetime(2026, 9, 2, 16, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000067'), time=datetime.datetime(2026, 9, 2, 8, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.0001'), time=datetime.datetime(2026, 9, 2, 0, 0, tzinfo=datetime.time

### `funding_payments` -- best-effort, unverifiable on this account

Same caveat as `market/classic.ipynb`: no dedicated "funding fee" endpoint was found
under `uta.*`. `account.financial_records`'s free-text `type` field is filtered here for
values containing `'funding'`, matching the substring guess used for Classic's
`futureTaxType`. Walking this account's whole reachable ledger (the endpoint refuses a
start time more than ~90 days back) turned up only four distinct values --
`ORDER_DEALT_IN`, `ORDER_DEALT_FROZEN_OUT`, `BUY_DEAL`, `SELL_DEAL` -- none containing
`'funding'`, and note they are upper-case, which is why the filter lower-cases before
matching. The filter remains unconfirmed.

In [15]:
async def funding_payments(symbol: str, start: datetime, end: datetime) -> list[FundingPayment]:
  # `financial_records` has no `symbol` param -- filtered client-side on the response's
  # own `symbol` field instead.
  raw = await client.uta.account.financial_records(
    category=PERP_CATEGORY, start_time=start, end_time=end,
  )
  return [
    FundingPayment(amount=-Decimal(r['amount']), time=r['ts'])
    for r in (raw['list'] or [])
    if r['symbol'] == symbol and 'funding' in r['type'].lower()
  ]

{symbol: await funding_payments(symbol, funding_start, funding_end) for symbol in PERP_MARKETS}

{'BTCUSDT': [], 'ETHUSDT': [], 'SOLUSDT': []}

In [16]:
async def perp_position(symbol: str) -> PerpPosition:
  raw = await client.uta.position.current_positions(category=PERP_CATEGORY, symbol=symbol)
  rows = raw['list'] or []
  row = rows[0] if rows else None
  if row is None:
    return PerpPosition()
  total = Decimal(row['total'])
  size = total if row['posSide'] == 'long' else -total
  return PerpPosition(size=size, entry_price=Decimal(row['avgPrice']))

{symbol: await perp_position(symbol) for symbol in PERP_MARKETS}

{'BTCUSDT': PerpPosition(size=Decimal('0'), entry_price=Decimal('0')),
 'ETHUSDT': PerpPosition(size=Decimal('0'), entry_price=Decimal('0')),
 'SOLUSDT': PerpPosition(size=Decimal('0'), entry_price=Decimal('0'))}

`perp_collateral()` reads the *same* unified `account.assets()` call as spot `position()`
above and as `reporting/uta.ipynb`'s `Snapshots` -- UTA has one collateral pool shared
across every category, so there's no per-symbol account lookup the way Classic's
`mix.account.get(symbol=...)` needs. `margin_mode` is read off `account.settings()`'s
`symbolConfigList`, which holds one row per (symbol, margin mode) rather than one per
symbol -- BTCUSDT appears twice live here, `crossed` at leverage 20 and `isolated` at
leverage 10, while ETHUSDT and SOLUSDT have no row at all. The lookup below takes the
first row matching the symbol and falls back to `'cross'` when there is none, then
translates Bitget's `'crossed'`/`'isolated'` spelling to the abstract
`Literal['cross', 'isolated']`.

In [17]:
async def perp_collateral(symbol: str) -> PerpCollateral:
  assets, settings = await asyncio.gather(
    client.uta.account.assets(), client.uta.account.settings(),
  )
  cfg = next((c for c in settings['symbolConfigList'] if c['symbol'] == symbol), None)
  margin_mode = 'isolated' if cfg and cfg['marginMode'] == 'isolated' else 'cross'
  return PerpCollateral(
    equity=assets['accountEquity'],
    free_collateral=assets['effEquity'],
    initial_margin=assets['imr'],
    maintenance_margin=assets['mmr'],
    leverage=assets['leverage'],
    margin_mode=margin_mode,
  )

{symbol: await perp_collateral(symbol) for symbol in PERP_MARKETS}

{'BTCUSDT': PerpCollateral(equity=Decimal('20.52335754'), free_collateral=Decimal('15.76141107'), initial_margin=Decimal('0'), maintenance_margin=Decimal('0'), leverage=Decimal('0'), margin_mode='cross'),
 'ETHUSDT': PerpCollateral(equity=Decimal('20.52308023'), free_collateral=Decimal('15.76144285'), initial_margin=Decimal('0'), maintenance_margin=Decimal('0'), leverage=Decimal('0'), margin_mode='cross'),
 'SOLUSDT': PerpCollateral(equity=Decimal('20.52280724'), free_collateral=Decimal('15.76163652'), initial_margin=Decimal('0'), maintenance_margin=Decimal('0'), leverage=Decimal('0'), margin_mode='cross')}

In [18]:
async def perp_available_notional(symbol: str) -> Decimal:
  collateral, rules = await asyncio.gather(perp_collateral(symbol), perp_rules(symbol))
  max_leverage = rules.details.get('maxLeverage') or Decimal(1)
  return collateral.free_collateral * max_leverage

{symbol: await perp_available_notional(symbol) for symbol in PERP_MARKETS}

{'BTCUSDT': Decimal('2364.24547800'),
 'ETHUSDT': Decimal('2364.26776950'),
 'SOLUSDT': Decimal('1576.18047500')}

In [ ]:
async def perp_place_order(symbol: str, order: Order, *, settings: Settings = {}) -> OrderResponse:
  qty_signed = Decimal(str(order['qty']))
  side = 'buy' if qty_signed > 0 else 'sell'
  qty = abs(qty_signed)
  price = Decimal(str(order['price']))
  body: LimitOrderRequest | MarketOrderRequest
  if order['type'] == 'MARKET':
    body = {'category': PERP_CATEGORY, 'symbol': symbol, 'side': side, 'qty': qty, 'orderType': 'market'}
  else:
    time_in_force = 'post_only' if order['type'] == 'POST_ONLY' else 'gtc'
    body = {
      'category': PERP_CATEGORY, 'symbol': symbol, 'side': side, 'qty': qty,
      'orderType': 'limit', 'price': price, 'timeInForce': time_in_force,
    }
  raw = await client.uta.trade.order.place(body)
  return OrderResponse(id=raw['orderId'], details=raw)

# Not executed here -- would place a real order on the account.
await perp_place_order('BTCUSDT', {'qty': Decimal('0.001'), 'price': Decimal('20000'), 'type': 'LIMIT'})

In [ ]:
async def perp_cancel_order(symbol: str, id: str, *, settings: Settings = {}):
  return await client.uta.trade.order.cancel(order_id=id, category=PERP_CATEGORY)

# Not executed here -- would cancel a real order on the account.
await perp_cancel_order('BTCUSDT', '123456')

### Coverage assessment: `PerpMarket` (UTA)

**Mostly supported**, and notably richer than Classic on the collateral side: UTA's
`account.assets()` directly exposes `imr`/`mmr`/`leverage`/`effEquity`, so
`perp_collateral()` needs no approximation the way Classic's does (see
`market/classic.ipynb`'s coverage note). With no open position on this account all four
read as `0` except `effEquity` (15.76 against 20.39 of equity), so the mapping is
demonstrated rather than numerically exercised. `index()` prefers `tickers`'s
`indexPrice`, which is present for futures categories (79426.616 for BTCUSDT live),
over the order-book-midpoint fallback Classic needs. `funding_payments` remains
best-effort/unverified for the same reason as Classic's.

One real upstream bug surfaced while building `next_funding()`:
`uta.market.funding_rate.current` raises a hard `ValidationError` for every symbol tried,
because `CurrentFundingRate.cashDividendNextUpdate` is typed
`NotRequired[TimestampMillis | None]` while the live API sends the literal string
`'null'`. Worked around above by reading that one call unvalidated and converting the
three mapped fields by hand.